In [ ]:
import sys
import os
import pandas as pd
import numpy as np

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from scripts.data_loading import consumers, accounts, transactions, category_mapping
from scripts.backfill_transactions import build_backfill_df

df = build_backfill_df()
df.head()

,prism_consumer_id,date,balance,credit_or_debit,amount_change,DQ_TARGET
0,3023,2021-08-31,225.95,starting value,0.00,0.0
1,3023,2021-03-24,205.32,DEBIT,20.63,0.0
2,3023,2021-03-27,445.32,CREDIT,240.00,0.0
3,3023,2021-03-27,785.32,CREDIT,340.00,0.0
4,3023,2021-03-29,760.32,DEBIT,25.00,0.0


## Feature Creation

In [4]:
# ---- end-of-day series (one point per day) ----
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(["prism_consumer_id", "date"])
df["day"] = df["date"].dt.normalize()

rows_daily = (
    df.groupby(["prism_consumer_id", "day"], as_index=False)
           .agg(
               balance=("balance", "last"),              # end-of-day balance
               DQ_TARGET=("DQ_TARGET", "max"),           # label
               amount_change=("amount_change", "sum"),   # net daily change (optional)
               n_tx=("amount_change", "size")            # number of intraday rows (optional)
           )
           .rename(columns={"day": "date"})
)


In [ ]:

rows_daily["date"] = pd.to_datetime(rows_daily["date"])
rows_daily = rows_daily.sort_values(["prism_consumer_id", "date"])

### Helper Functions

In [6]:
def _safe_div(a, b):
    return a / (b + 1e-9)

def _pct(x, thresh):
    return float((x < thresh).mean()) if len(x) else np.nan

def _trend(y):
    """Slope of y over time index (simple linear fit)."""
    y = np.asarray(y, dtype=float)
    if len(y) < 2:
        return np.nan
    x = np.arange(len(y), dtype=float)
    return float(np.polyfit(x, y, 1)[0])

def _last_window(df, days):
    if df.empty:
        return df
    cutoff = df["date"].max() - pd.Timedelta(days=days)
    return df[df["date"] >= cutoff]

def _window_stats(g, col, days, prefix):
    d = _last_window(g, days)
    s = d[col]
    out = {
        f"{prefix}__mean__{days}d": float(s.mean()) if len(s) else np.nan,
        f"{prefix}__median__{days}d": float(s.median()) if len(s) else np.nan,
        f"{prefix}__min__{days}d": float(s.min()) if len(s) else np.nan,
        f"{prefix}__max__{days}d": float(s.max()) if len(s) else np.nan,
        f"{prefix}__std__{days}d": float(s.std()) if len(s) else np.nan,
        f"{prefix}__trend__{days}d": _trend(s.values),
    }
    return out

def _window_counts(g, days, prefix):
    d = _last_window(g, days)
    out = {
        f"{prefix}__n_days__{days}d": int(d["date"].nunique()) if len(d) else 0,
        f"{prefix}__n_tx__{days}d": int(d["n_tx"].sum()) if len(d) else 0,
    }
    return out

### Balance Features

In [7]:
bal_all = rows_daily.groupby("prism_consumer_id").agg(
    balance__mean__all=("balance", "mean"),
    balance__median__all=("balance", "median"),
    balance__min__all=("balance", "min"),
    balance__max__all=("balance", "max"),
    balance__std__all=("balance", "std"),
    balance__pct_negative__all=("balance", lambda x: (x < 0).mean()),
    balance__pct_below_100__all=("balance", lambda x: (x < 100).mean()),
    balance__pct_below_500__all=("balance", lambda x: (x < 500).mean()),
    n_days__all=("date", "nunique"),
)

### Daily Features

In [10]:
def window_daily_features(df, days):
    max_date = df.groupby("prism_consumer_id")["date"].max()
    tmp = df.join(max_date.rename("max_date"), on="prism_consumer_id")
    tmp = tmp[tmp["date"] >= (tmp["max_date"] - pd.Timedelta(days=days))]

    out = tmp.groupby("prism_consumer_id").agg(
        **{
            f"balance__mean__{days}d": ("balance", "mean"),
            f"balance__min__{days}d": ("balance", "min"),
            f"balance__std__{days}d": ("balance", "std"),
            f"balance__pct_negative__{days}d": ("balance", lambda x: (x < 0).mean()),
            f"cashflow__net__{days}d": ("amount_change", "sum"),
            f"cashflow__mean_daily__{days}d": ("amount_change", "mean"),
            f"cashflow__volatility__{days}d": ("amount_change", "std"),
            f"n_tx__{days}d": ("n_tx", "sum"),
            f"n_days__{days}d": ("date", "nunique"),
        }
    )
    return out

daily_30  = window_daily_features(rows_daily, 30)
daily_60  = window_daily_features(rows_daily, 60)
daily_90  = window_daily_features(rows_daily, 90)
daily_180 = window_daily_features(rows_daily, 180)

tx = df.copy()
tx["date"] = pd.to_datetime(tx["date"])

tx["is_credit"] = tx["credit_or_debit"].eq("CREDIT")
tx["is_debit"]  = tx["credit_or_debit"].eq("DEBIT")

# totals + counts
tx_all = tx.groupby("prism_consumer_id").agg(
    tx__n__all=("amount_change", "size"),
    tx__std_amount__all=("amount_change", "std"),
    credit__total__all=("amount_change", lambda x: x[x > 0].sum()),
    debit__total__all=("amount_change", lambda x: np.abs(x[x < 0]).sum()),
    tx__max_credit__all=("amount_change", lambda x: x[x > 0].max() if (x > 0).any() else 0.0),
    tx__max_debit__all=("amount_change", lambda x: np.abs(x[x < 0]).max() if (x < 0).any() else 0.0),
)

tx_all["credit_debit__ratio__all"] = tx_all["credit__total__all"] / (tx_all["debit__total__all"] + 1e-9)


## Category Features

In [15]:
cat_map = category_mapping.rename(
    columns={"category_id": "category", "category": "category_name"}
)

txc = transactions.merge(cat_map, on="category", how="left").copy()
txc["posted_date"] = pd.to_datetime(txc["posted_date"])
txc["signed_amount"] = np.where(txc["credit_or_debit"].eq("CREDIT"),
                                txc["amount"].astype("float32"),
                                -txc["amount"].astype("float32"))

TOPK = 30
top_cats = txc["category"].value_counts().head(TOPK).index
txc = txc[txc["category"].isin(top_cats)]

# all-time
cat_all = (
    txc.groupby(["prism_consumer_id", "category"])
       .agg(cat_net_total=("signed_amount", "sum"),
            cat_n=("signed_amount", "count"))
       .unstack(fill_value=0)
)
cat_all.columns = [f"cat_{int(c)}__{stat}__all" for stat, c in cat_all.columns]

# 90d window
max_date = txc.groupby("prism_consumer_id")["posted_date"].max()
tmp = txc.join(max_date.rename("max_date"), on="prism_consumer_id")
tmp = tmp[tmp["posted_date"] >= (tmp["max_date"] - pd.Timedelta(days=90))]

cat_90 = (
    tmp.groupby(["prism_consumer_id", "category"])
       .agg(cat_net_total=("signed_amount", "sum"),
            cat_n=("signed_amount", "count"))
       .unstack(fill_value=0)
)
cat_90.columns = [f"cat_{int(c)}__{stat}__90d" for stat, c in cat_90.columns]

In [16]:
INCOME_CATS = {
    2,   # DEPOSIT
    3,   # PAYCHECK
    6,   # REFUND
    7,   # INVESTMENT_INCOME
    8,   # OTHER_BENEFITS
    9,   # UNEMPLOYMENT_BENEFITS
    42,  # GOVERNMENT_SERVICES
    45,  # INVESTMENT
    49,  # PENSION
}

ESSENTIAL_CATS = {
    11,  # TAX
    12,  # LOAN
    13,  # INSURANCE
    17,  # AUTOMOTIVE
    18,  # GROCERIES
    22,  # ESSENTIAL_SERVICES
    23,  # ACCOUNT_FEES
    26,  # CREDIT_CARD_PAYMENT
    27,  # HEALTHCARE_MEDICAL
    29,  # EDUCATION
    31,  # BILLS_UTILITIES
    32,  # MORTGAGE
    33,  # CHILD_DEPENDENTS
    34,  # RENT
    36,  # AUTO_LOAN
    38,  # DEBT
}

DISCRETIONARY_CATS = {
    14,  # FOOD_AND_BEVERAGES
    16,  # GENERAL_MERCHANDISE
    19,  # ATM_CASH
    20,  # ENTERTAINMENT
    21,  # TRAVEL
    24,  # HOME_IMPROVEMENT
    28,  # PETS
    30,  # GIFTS_DONATIONS
    35,  # BNPL
    39,  # FITNESS
    40,  # TRANSPORATION
    46,  # GAMBLING
    48,  # TIME_OR_STUFF
}

In [17]:
g = txc.groupby("prism_consumer_id")

income_total = (
    txc[txc["category"].isin(INCOME_CATS)]
    .groupby("prism_consumer_id")["signed_amount"]
    .apply(lambda x: x[x > 0].sum())
    .rename("income__total__all")
)

ess_total = (
    txc[txc["category"].isin(ESSENTIAL_CATS)]
    .groupby("prism_consumer_id")["signed_amount"]
    .apply(lambda x: np.abs(x[x < 0]).sum())
    .rename("essentials_spend__total__all")
)

disc_total = (
    txc[txc["category"].isin(DISCRETIONARY_CATS)]
    .groupby("prism_consumer_id")["signed_amount"]
    .apply(lambda x: np.abs(x[x < 0]).sum())
    .rename("discretionary_spend__total__all")
)

group_feats = pd.concat([income_total, ess_total, disc_total], axis=1).fillna(0)
group_feats["essentials__pct_of_income__all"] = group_feats["essentials_spend__total__all"] / (group_feats["income__total__all"] + 1e-9)
group_feats["discretionary__pct_of_income__all"] = group_feats["discretionary_spend__total__all"] / (group_feats["income__total__all"] + 1e-9)

### Results

In [18]:
X = (
    bal_all
    .join([daily_30, daily_60, daily_90, daily_180], how="outer")
    .join(tx_all, how="outer")
    .join(cat_all, how="outer")
    .join(cat_90, how="outer")
    .join(group_feats, how="outer")
).replace([np.inf, -np.inf], np.nan).fillna(0)

y = (
    rows_daily
    .groupby("prism_consumer_id")["DQ_TARGET"]
    .max()
    .rename("DQ_TARGET")
)

features_df = X.join(y, how="inner")
print("features_df shape:", features_df.shape, "num_features:", features_df.shape[1]-1)
features_df.head()

features_df shape: (12900, 178) num_features: 177


,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,balance__mean__30d,...,cat_39__cat_n__90d,cat_40__cat_n__90d,cat_45__cat_n__90d,cat_46__cat_n__90d,income__total__all,essentials_spend__total__all,discretionary_spend__total__all,essentials__pct_of_income__all,discretionary__pct_of_income__all,DQ_TARGET
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,276.961538,70.09,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,-497.176071,...,3.0,1.0,0.0,0.0,9340.520508,1718.160034,7332.270020,0.183947,0.784996,0.0
1,1674.533585,1758.35,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,2671.932727,...,0.0,2.0,0.0,0.0,13414.009766,680.210022,11539.479492,0.050709,0.860256,0.0
10,-106.435115,-98.40,-1108.49,929.25,501.726601,0.595420,0.633588,0.839695,131.0,-382.525455,...,0.0,1.0,0.0,4.0,15513.070312,1527.850220,9379.980469,0.098488,0.604650,0.0
100,-3231.228909,-3752.93,-6273.18,802.40,2080.213280,0.963636,0.963636,0.981818,55.0,-4166.195000,...,0.0,0.0,0.0,0.0,24423.531250,18534.431641,200.000000,0.758876,0.008189,0.0
1000,1013.427875,615.39,-22.85,12589.57,1545.044777,0.025000,0.112500,0.437500,80.0,601.327692,...,0.0,0.0,1.0,0.0,58994.343750,17348.220703,0.000000,0.294066,0.000000,0.0


In [19]:
X = features_df.drop(columns=["DQ_TARGET"]).select_dtypes(include=[np.number]).copy()

summary = pd.DataFrame({
    "feature": X.columns,
    "nonzero_rate": (X != 0).mean().values,
    "mean": X.mean().values,
    "std": X.std().values,
})

summary = summary.sort_values("nonzero_rate", ascending=False)
summary.head(30)

,feature,nonzero_rate,mean,std
25,n_tx__60d,1.000000,1.659543e+02,1.358370e+02
16,n_tx__30d,1.000000,8.647891e+01,7.113584e+01
45,tx__n__all,1.000000,4.041308e+02,3.676904e+02
44,n_days__180d,1.000000,8.225372e+01,4.682480e+01
43,n_tx__180d,1.000000,3.585608e+02,2.945833e+02
35,n_days__90d,1.000000,5.210047e+01,2.273709e+01
34,n_tx__90d,1.000000,2.415682e+02,1.970079e+02
26,n_days__60d,1.000000,3.567775e+01,1.520348e+01
8,n_days__all,1.000000,9.172225e+01,5.621988e+01
17,n_days__30d,1.000000,1.864915e+01,7.663812e+00


In [20]:
feature_cols = [c for c in features_df.columns if c != "DQ_TARGET"]

groups = {
    "Balance behavior": [c for c in feature_cols if c.startswith("balance__")],
    "Cashflow behavior": [c for c in feature_cols if c.startswith("cashflow__")],
    "Transaction activity": [c for c in feature_cols if c.startswith("tx__") or c.startswith("credit_") or c.startswith("debit_")],
    "Category-level behavior": [c for c in feature_cols if c.startswith("cat_")],
    "Income & spending burden": [c for c in feature_cols if c.startswith("income__") or "spend" in c or "pct_of_income" in c],
}

for title, cols in groups.items():
    print(f"\n{title} ({len(cols)} features):")
    for c in cols:
        print("  -", c)


Balance behavior (24 features):
  - balance__mean__all
  - balance__median__all
  - balance__min__all
  - balance__max__all
  - balance__std__all
  - balance__pct_negative__all
  - balance__pct_below_100__all
  - balance__pct_below_500__all
  - balance__mean__30d
  - balance__min__30d
  - balance__std__30d
  - balance__pct_negative__30d
  - balance__mean__60d
  - balance__min__60d
  - balance__std__60d
  - balance__pct_negative__60d
  - balance__mean__90d
  - balance__min__90d
  - balance__std__90d
  - balance__pct_negative__90d
  - balance__mean__180d
  - balance__min__180d
  - balance__std__180d
  - balance__pct_negative__180d

Cashflow behavior (12 features):
  - cashflow__net__30d
  - cashflow__mean_daily__30d
  - cashflow__volatility__30d
  - cashflow__net__60d
  - cashflow__mean_daily__60d
  - cashflow__volatility__60d
  - cashflow__net__90d
  - cashflow__mean_daily__90d
  - cashflow__volatility__90d
  - cashflow__net__180d
  - cashflow__mean_daily__180d
  - cashflow__volatility

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# Prepare data
df = features_df.replace([np.inf, -np.inf], np.nan)
df = df[df["DQ_TARGET"].notna()].copy()

y = df["DQ_TARGET"].astype(int)
X = (
    df.drop(columns=["DQ_TARGET"])
      .select_dtypes(include=[np.number])
      .fillna(0)
)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Tree-based gradient boosting (handles nonlinearity)
model = HistGradientBoostingClassifier(
    max_depth=6,
    max_iter=300,
    learning_rate=0.05,
    min_samples_leaf=50,
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate
val_prob = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_prob)

print("HistGradientBoosting AUC:", round(auc, 4))


HistGradientBoosting AUC: 0.7953
